# Section 1: Problem and Population

The subject would be a person who is out of a home and has no consistent source of income. The failure point is the lack of a point to action, there is no way from this scenario to reasonably find a source of income by yourself. This falls under goal 11 for sustainable cities and communities. The subject was modified so that rather than just a person who is homeless they also need to have no consistent source of income so that we target the people that need the assistance the most. The failure point was also expanded to account for the change in the subject.

# Section 2: Proposed System

You would first receive a response from a caller about their situation, then the AI would analyze the information provided. Preferably the information contains their current living situation, how many resources they have, and a future point of contact for further support. Then the AI would prompt a response and send the call to the appropriate service and categorize the urgency of the person's situation. They would then receive help from that service to find a route to a stable source of income, and possible permanent housing.

# Section 3: Project Code

In [ ]:
!pip install -q google-generativeai

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import json
import time

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini initialized successfully.")

Gemini initialized successfully.


In [ ]:
import re
from google.api_core.exceptions import TooManyRequests

Imported Gemini AI

In [ ]:
def get_completion(prompt, system_instruction="You are a helpful assistant.", max_retries=3):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=system_instruction
    )

    retries = 0
    while retries < max_retries:
        try:
            response = m.generate_content(prompt)
            time.sleep(12)
            return response.text
        except TooManyRequests as e:
            retries += 1
            if retries == max_retries:
                raise e

            match = re.search(r'Please retry in (\d+\.?\d*)s\.', str(e))
            wait_time = 60
            if match:
                wait_time = float(match.group(1))

            print(f"Quota exceeded. Retrying in {wait_time:.2f} seconds... (Attempt {retries}/{max_retries})")
            time.sleep(wait_time + 1)
        except Exception as e:
            raise e

    return ""

Code to limit the amount of packets being sent to mitigate rate limiting from the Ai

In [ ]:
multilingual_prompt_active = """
You are a helpful assistant for the San Jose 311 civic reporting system.
A person has submitted an assistance request. Write a brief, polite acknowledgment
that: (1) confirms the request was received, (2) names the situation they reported,
and (3) tells them the expected response timeframe (3–5 business days for
non-emergency issues). Keep the response under 80 words.
Detect the language the resident wrote in and respond in that same language.
"""

test_messages = {
    "Spanish": "Hay basura ilegal tirada detrás de mi edificio en la calle Oak. Por favor, que alguien venga a limpiar.",
    "Vietnamese": "Có rác thải bất hợp pháp bị vứt sau tòa nhà của tôi trên đường Oak. Xin hãy cử người đến dọn dẹp.",
    "Cantonese": "我大廈後面嘅橡樹街有人非法棄置垃圾。請派人嚟清理。"
    # still needs to be changed
}

for language, message in test_messages.items():
    print(f"=== {language} ===")
    print(f"Input:  {message}")
    result = get_completion(message, multilingual_prompt_active)
    print(f"Output: {result}")
    print()

=== Spanish ===
Input:  Hay basura ilegal tirada detrás de mi edificio en la calle Oak. Por favor, que alguien venga a limpiar.
Output: Gracias por contactar San Jose 311. Hemos recibido su reporte de basura ilegal detrás de su edificio en la calle Oak. Un equipo evaluará la situación y tomará las medidas adecuadas. Puede esperar una respuesta o acción dentro de 3 a 5 días hábiles. Agradecemos su ayuda para mantener limpia nuestra ciudad.

=== Vietnamese ===
Input:  Có rác thải bất hợp pháp bị vứt sau tòa nhà của tôi trên đường Oak. Xin hãy cử người đến dọn dẹp.
Output: Chúng tôi đã nhận được yêu cầu của quý vị về việc đổ rác thải bất hợp pháp trên đường Oak. Vấn đề này sẽ được chuyển đến đội ngũ của chúng tôi. Quý vị có thể mong đợi phản hồi trong vòng 3-5 ngày làm việc. Cảm ơn quý vị đã báo cáo!

=== Cantonese ===
Input:  我大廈後面嘅橡樹街有人非法棄置垃圾。請派人嚟清理。
Output: 您好！我們已收到您關於橡樹街非法棄置垃圾的報告。我們會安排人員處理。非緊急情況一般會在 3–5 個工作天內處理完畢。感謝您的報告！



This code block would receive a call from a caller and then give a response based on the call and give the caller an expected response time from the appropriate service

In [ ]:
schema_prompt = """
Extract information from this 311 resident complaint.
Return ONLY valid JSON with exactly these four fields:
{
  "gen_info": string (their current living situation and what kind of resources they have),
  "urgency": "LOW" or "MEDIUM" or "HIGH",
  "department": string (which San Jose city department should respond: e.g. "Environmental Services", "Public Works", "Code Enforcement", "Parks & Recreation"),
  "resident_language": string (language the resident wrote in, e.g. "English", "Spanish", "Vietnamese")
}
Urgency guide: LOW = no need to receive help, MEDIUM = current situation requires attention within 1 month, HIGH = caller requires immediate assistance.
No explanation. No markdown. JSON only.
"""

print("Schema defined. The AI must return exactly these four fields:")
for field in [ "gen_info", "urgency", "department", "resident_language"]:
    print(f"  - {field}")

Schema defined. The AI must return exactly these four fields:
  - gen_info
  - urgency
  - department
  - resident_language


Defines a schema which will the AI will use to sort data. In this case it will sort the provided data into 4 different fields: gen_info, urgency, department, and resident_language.

In [ ]:
resident_message = "I am currently out of a home and have no money on me."

In [ ]:
def extract_structured(message):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=schema_prompt
    )
    response = m.generate_content(message)
    time.sleep(12)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

print("--- Resident message ---")
print(resident_message)
print("\n--- Structured extraction (run 3 times — format is identical each time) ---")

for i in range(1, 4):
    print(f"\nRun {i}:")
    result = extract_structured(resident_message)
    print(json.dumps(result, indent=2, ensure_ascii=False))

--- Resident message ---
I am currently out of a home and have no money on me.

--- Structured extraction (run 3 times — format is identical each time) ---

Run 1:
{
  "gen_info": "I am currently out of a home and have no money on me.",
  "urgency": "HIGH",
  "department": "Housing and Community Services",
  "resident_language": "English"
}

Run 2:
{
  "gen_info": "Currently homeless and has no money.",
  "urgency": "HIGH",
  "department": "Social Services",
  "resident_language": "English"
}

Run 3:
{
  "gen_info": "Out of a home and have no money on me.",
  "urgency": "HIGH",
  "department": "Housing Department",
  "resident_language": "English"
}


Defines the schema as the instructions that the AI will use to sort data. Also formats the output to be neater and runs up to 3 messages at once.

In [ ]:
test_messages = [
    {
        "label": "Scenario 1",
        "message": "I have a home but I am at risk of getting evicted."
                   "I'm worried for my own safety and have no clue how to proceed."
    },
    {
        "label": "Scenario 2 (Spanish)",
        "message": "Actualmente no tengo casa ni dinero, pero aun así vivo bastante cómodamente."
                   "Me gustaría encontrar una fuente de ingresos estable y agradecería ayuda para conseguirla."
    },
    {
        "label": "Scenario 3",
        "message": "Im currently stuck out on the street with nothing to my name. "
                   "I am also currently with other people but they don't seem very friendly. "
                   "If there is anything I can do to make money I would like that option."
    }
]

for item in test_messages:
    print(f"=== {item['label']} ===")
    print(f"Input: {item['message']}")
    result = extract_structured(item["message"])
    print("Output:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print()

=== Scenario 1 ===
Input: I have a home but I am at risk of getting evicted.I'm worried for my own safety and have no clue how to proceed.
Output:
{
  "gen_info": "Resident has a home but is at risk of eviction and is worried for their safety, unsure how to proceed.",
  "urgency": "HIGH",
  "department": "Housing Department",
  "resident_language": "English"
}

=== Scenario 2 (Spanish) ===
Input: Actualmente no tengo casa ni dinero, pero aun así vivo bastante cómodamente.Me gustaría encontrar una fuente de ingresos estable y agradecería ayuda para conseguirla.
Output:
{
  "gen_info": "Actualmente no tengo casa ni dinero, pero aun así vivo bastante cómodamente.",
  "urgency": "MEDIUM",
  "department": "Parks, Recreation and Neighborhood Services",
  "resident_language": "Spanish"
}

=== Scenario 3 ===
Input: Im currently stuck out on the street with nothing to my name. I am also currently with other people but they don't seem very friendly. If there is anything I can do to make money I 

This code block reads the provided messages and sorts the information provided using the schema as the base. General info is pulled from the message and the AI makes the call about how urgent the current situation of the caller is, which department the message should be forwarded to, and which language the message is sent in.

# Section 4: Edge Case Elicitation

In [ ]:
resident_message= {"I live in a home with my family, we are well off and have no issues with money."
"However I would like to help one of my friends and am calling on behalf of them since they have no way to ask for help themselves."
"They currently are out of a home and have little to no money and are looking for a job."}

In [ ]:
def extract_structured(message):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=schema_prompt
    )
    response = m.generate_content(message)
    time.sleep(12)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

print("--- Resident message ---")
print(resident_message)
print("\n--- Structured extraction (run 3 times — format is identical each time) ---")

for i in range(1, 2):
    print(f"\nRun {i}:")
    result = extract_structured(resident_message)
    print(json.dumps(result, indent=2, ensure_ascii=False))

--- Resident message ---
{'I live in a home with my family, we are well off and have no issues with money.However I would like to help one of my friends and am calling on behalf of them since they have no way to ask for help themselves.They currently are out of a home and have little to no money and are looking for a job.'}

--- Structured extraction (run 3 times — format is identical each time) ---

Run 1:
{
  "gen_info": "Friend is out of a home, has little to no money, and is looking for a job.",
  "urgency": "HIGH",
  "department": "Housing and Community Development",
  "resident_language": "English"
}


The prompt uses a case where someone else calls on the subjects behalf.

The AI seemingly prioritzes the information of the caller rather than the subject who actually needs help.

This is a miss, the AI completely misses the actual subject and reads the information of the caller instead.